# Reason based data 

## DB Connection and data retrieval

dbname : fishdata

In [ ]:
import psycopg2
import pandas as pd
conn = psycopg2.connect(
    dbname="fishdata",
    user="fishdata",
    password="Jalkirani",
    host="fishdata-collection.c5gmcqq8k94k.ap-south-1.rds.amazonaws.com",
    port="5432"
)
cur = conn.cursor()
cur.execute("SELECT * FROM fishinfo")
rows = cur.fetchall()
column_names = [desc[0] for desc in cur.description]
df = pd.DataFrame(rows, columns=column_names)
cur.close()
conn.close()
df.head()

,id,type,labels,description,image_url
0,77,tuna,good,None,https://fish-data-collection.s3.ap-south-1.ama...
1,79,tuna,bad,shreyas demo,https://fish-data-collection.s3.ap-south-1.ama...
2,80,rohu,good,None,https://fish-data-collection.s3.ap-south-1.ama...
3,81,mackerel,ok,None,https://fish-data-collection.s3.ap-south-1.ama...
4,82,mackerel,good,None,https://fish-data-collection.s3.ap-south-1.ama...


In [ ]:
df.tail()

,id,type,labels,description,image_url
81676,89938,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
81677,89939,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
81678,89940,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
81679,89941,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
81680,89942,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....


In [ ]:
df.shape

(81681, 5)

In [ ]:
df.dtypes

id              int64
type           object
labels         object
description    object
image_url      object
dtype: object

## Data cleaning

In [ ]:
# Drop rows with missing values in 'description' column
df.dropna(subset=['description'], inplace=True)

In [ ]:
df.shape

(23940, 5)

In [ ]:
df['type'].unique()

array(['tuna                ', 'mackerel            ',
       'katla               ', 'rohu                ',
       'indian salmon       ', 'trial_purpose       ',
       'trail_purpose       ', 'trial               ',
       'white prawns        ', 'sardine             ',
       'test                ', 'pink perch          ',
       'roopchand           ', 'tilapia             ',
       'tiger prawns        ', 'barracuda           ',
       'travely             ', 'white pomfret       ',
       'seer                ', 'red prawns          ',
       'croaker             ', 'sea prawns          ',
       'lady                ', 'c boss              ',
       'mullet              ', 'blue crab           ',
       'hilisa              ', 'dotted crab         '], dtype=object)

In [ ]:
import re

# Function to remove extra spaces for string columns
def remove_extra_spaces(column):
    if column.dtype == 'object':
        return column.apply(lambda x: re.sub(r'\s+', ' ', x).strip())
    return column

# Apply the function to all columns except the first column
df.iloc[:, 1:] = df.iloc[:, 1:].apply(remove_extra_spaces)


In [ ]:
df['type'].unique()

array(['tuna', 'mackerel', 'katla', 'rohu', 'indian salmon',
       'trial_purpose', 'trail_purpose', 'trial', 'white prawns',
       'sardine', 'test', 'pink perch', 'roopchand', 'tilapia',
       'tiger prawns', 'barracuda', 'travely', 'white pomfret', 'seer',
       'red prawns', 'croaker', 'sea prawns', 'lady', 'c boss', 'mullet',
       'blue crab', 'hilisa', 'dotted crab'], dtype=object)

In [ ]:
df['labels'].unique()

array(['bad', 'good', 'ok'], dtype=object)

In [ ]:
df['description'].unique()

array(['shreyas demo', '5 fish', 'katla', 'katala', 'basa',
       'trial_purpose', 'demo', 'bekar taste hai', 'Softness', 'Test',
       'a', 'à', '', 'i9o ypoppuppp0p0uuuuup', 'l', 'w', 'Tuna Fish',
       'Good', 'Red Head in Prawns', 'o', 'up',
       'Softness, Cuts and Damage', 'Softness, Red Head in Prawns',
       'Cuts and Damage', 'Size', 'hi', 'tr', 'Cuts and Damage, Softness',
       'Smell', 'Red Head in Prawns, Softness', 'Others, Softness',
       'Softness, Smell', 'Color, Smell', 'Size, Red Head in Prawns',
       'Red Head in Prawns, Cuts and Damage',
       'Cuts and Damage, Red Head in Prawns', 'Smell, Softness',
       'Softness, Color', 'Softness, Red Head in Prawns, Smell',
       'Softness, Size', 'Softness, Cuts and Damage, Smell', 'Color',
       'Softness, Color, Red Head in Prawns',
       'Cuts and Damage, Softness, Red Head in Prawns', 'Color, Softness',
       'Size, Softness', 'Smell, Color, Softness',
       'Softness, Cuts and Damage, Size',
       'Cu

In [ ]:
# Filter the DataFrame to keep rows where 'description' doesn't contain any of the strings
filter_strings = ['demo', 'test', 'trial_purpose', 'trial', 'trail_purpose']
c1 = df['type'].str.contains('|'.join(filter_strings))

# Filter the DataFrame to keep rows where 'labels' doesn't contain 'ok  '
c2 = df['labels']=='ok'

# Filter the DataFrame to keep rows where 'description' doesn't contain any of the strings
filter_strings = ['shreyas demo', '5 fish', 'Demo', 'katla', 'katala', 'basa',
              'trial_purpose', 'demo', 'bekar taste hai', 'Test', 'a', 'à', '',
              'i9o ypoppuppp0p0uuuuup', 'l', 'w', 'Tuna Fish', 'Good', 'o', 'up',
              'hi', 'tr', ]
c3 = df['description'].apply(lambda x: any(item==x for item in filter_strings))

df = df[(~c1)&(~c2)&(~c3)]

In [ ]:
df

,id,type,labels,description,image_url
3616,80310,mackerel,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
3626,81921,mackerel,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
3627,82094,mackerel,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
3631,82785,white prawns,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
3633,83128,white prawns,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
...,...,...,...,...,...
81676,89938,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
81677,89939,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
81678,89940,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....
81679,89941,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....


In [ ]:
df['type'].value_counts()

white prawns     9609
mackerel         7150
sardine          6286
tuna              250
sea prawns        193
seer               69
pink perch         67
tiger prawns       49
white pomfret      37
indian salmon      24
red prawns         24
dotted crab        20
barracuda          14
lady               12
mullet             10
blue crab           8
c boss              7
tilapia             6
croaker             5
hilisa              1
Name: type, dtype: int64

In [ ]:
df['image_url'].iloc[0]

'https://fish-data-collection-v2.s3.ap-south-1.amazonaws.com/mackerel/bad/20240910100835143_mackerel_bad.jpeg'

## saving the filtered DataFrame to a CSV file

In [ ]:
df.to_csv(r"testing_dataset2.csv", index=False)

In [ ]:
df.to_csv(r"testing_dataset2.csv", index=False)

## Loading saved data

In [ ]:
import pandas as pd
import os

In [ ]:
feedback_df = pd.read_csv(r"testing_dataset2.csv")
feedback_df['Filename'] = feedback_df['image_url'].apply(lambda x: os.path.basename(x))
feedback_df['Date'] = feedback_df['Filename'].apply(lambda x: os.path.splitext(x)[0].split('_')[0])
feedback_df['Date'] = feedback_df['Date'].apply(lambda x: f'{x[0:4]}-{x[4:6]}-{x[6:8]}')
# feedback_df.drop(['image_url'], axis=1, inplace=True)

In [ ]:
feedback_df

,id,type,labels,description,image_url,Filename,Date
0,80310,mackerel,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20240910100835143_mackerel_bad.jpeg,2024-09-10
1,81921,mackerel,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20240926094227777_mackerel_bad.jpeg,2024-09-26
2,82094,mackerel,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20241002102426458_mackerel_bad.jpeg,2024-10-02
3,82785,white prawns,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20241015103307962_white prawns_bad.jpeg,2024-10-15
4,83128,white prawns,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20241017101814512_white prawns_bad.jpeg,2024-10-17
...,...,...,...,...,...,...,...
23836,89938,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20250128113440816_pink perch_bad.jpeg,2025-01-28
23837,89939,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20250128113451297_pink perch_bad.jpeg,2025-01-28
23838,89940,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20250128113459764_pink perch_bad.jpeg,2025-01-28
23839,89941,pink perch,bad,Softness,https://fish-data-collection-v2.s3.ap-south-1....,20250128113508768_pink perch_bad.jpeg,2025-01-28


In [ ]:
feedback_df['type'].unique()

array(['mackerel', 'white prawns', 'tiger prawns', 'white pomfret',
       'pink perch', 'sardine', 'seer', 'red prawns', 'croaker', 'tuna',
       'indian salmon', 'barracuda', 'sea prawns', 'lady', 'c boss',
       'mullet', 'blue crab', 'hilisa', 'tilapia'], dtype=object)

In [ ]:
feedback_df['description'].unique()

array(['Softness', 'Red Head in Prawns', 'Softness, Cuts and Damage',
       'Softness, Red Head in Prawns', 'Cuts and Damage', 'Size',
       'Cuts and Damage, Softness', 'Smell',
       'Red Head in Prawns, Softness', 'Others, Softness',
       'Softness, Smell', 'Color, Smell', 'Size, Red Head in Prawns',
       'Red Head in Prawns, Cuts and Damage',
       'Cuts and Damage, Red Head in Prawns', 'Smell, Softness',
       'Softness, Color', 'Softness, Red Head in Prawns, Smell',
       'Softness, Size', 'Softness, Cuts and Damage, Smell', 'Color',
       'Softness, Color, Red Head in Prawns',
       'Cuts and Damage, Softness, Red Head in Prawns', 'Color, Softness',
       'Size, Softness', 'Smell, Color, Softness',
       'Softness, Cuts and Damage, Size',
       'Cuts and Damage, Size, Softness',
       'Softness, Cuts and Damage, Color', 'Red Head in Prawns, Size',
       'Softness, Size, Red Head in Prawns',
       'Softness, Cuts and Damage, Red Head in Prawns',
       'Cuts and

## Filtering data

In [ ]:
filtered_df = feedback_df[feedback_df['labels']=='bad']
filtered_df = filtered_df[filtered_df['description'].str.contains('Cuts')]

In [ ]:
filtered_df = filtered_df[filtered_df['type']=='sardine']

In [ ]:
filtered_df

,id,type,labels,description,image_url,Filename,Date
561,23594,sardine,bad,Cuts and Damage,https://fish-data-collection.s3.ap-south-1.ama...,20230916114452264_sardine_bad.jpeg,2023-09-16
562,23596,sardine,bad,Cuts and Damage,https://fish-data-collection.s3.ap-south-1.ama...,20230916114514979_sardine_bad.jpeg,2023-09-16
563,23601,sardine,bad,Cuts and Damage,https://fish-data-collection.s3.ap-south-1.ama...,20230916114612276_sardine_bad.jpeg,2023-09-16
643,23863,sardine,bad,Cuts and Damage,https://fish-data-collection.s3.ap-south-1.ama...,20230919123618572_sardine_bad.jpeg,2023-09-19
645,23865,sardine,bad,"Softness, Cuts and Damage",https://fish-data-collection.s3.ap-south-1.ama...,20230919123637784_sardine_bad.jpeg,2023-09-19
...,...,...,...,...,...,...,...
23079,85950,sardine,bad,"Softness, Cuts and Damage",https://fish-data-collection-v2.s3.ap-south-1....,20241203102411837_sardine_bad.jpeg,2024-12-03
23528,88356,sardine,bad,"Cuts and Damage, Softness",https://fish-data-collection-v2.s3.ap-south-1....,20250107115016759_sardine_bad.jpeg,2025-01-07
23533,88395,sardine,bad,"Softness, Cuts and Damage",https://fish-data-collection-v2.s3.ap-south-1....,20250107120121687_sardine_bad.jpeg,2025-01-07
23534,88399,sardine,bad,"Softness, Cuts and Damage",https://fish-data-collection-v2.s3.ap-south-1....,20250107120310894_sardine_bad.jpeg,2025-01-07


In [ ]:
filtered_df[(filtered_df['Date']>'2024-04-30')&(filtered_df['description'].str.lower().str.contains('size'))]

,id,type,labels,description,Filename,Date
10760,65863,sardine,bad,Size,20240525103614445_sardine_bad.jpeg,2024-05-25
18114,63008,sardine,bad,Size,20240502100653822_sardine_bad.jpeg,2024-05-02
18143,63037,sardine,bad,"Size, Softness",20240502101213407_sardine_bad.jpeg,2024-05-02
18160,63054,sardine,bad,"Size, Softness",20240502101604319_sardine_bad.jpeg,2024-05-02
19444,64464,sardine,bad,Size,20240514093027591_sardine_bad.jpeg,2024-05-14
19515,64534,sardine,bad,"Size, Softness",20240514094832469_sardine_bad.jpeg,2024-05-14
19561,64580,sardine,bad,"Cuts and Damage, Size, Softness",20240514095818001_sardine_bad.jpeg,2024-05-14
19944,65014,sardine,bad,"Softness, Size",20240516101021501_sardine_bad.jpeg,2024-05-16
19945,65015,sardine,bad,Size,20240516101030217_sardine_bad.jpeg,2024-05-16
19946,65016,sardine,bad,Size,20240516101040274_sardine_bad.jpeg,2024-05-16


In [ ]:
filtered_df = feedback_df[feedback_df['description'].str.contains('Softness')]
filtered_df = filtered_df[filtered_df['labels']=='bad']
filtered_df = filtered_df[filtered_df['type']=='sardine']

In [ ]:
filtered_df

,id,type,labels,description,Filename,Date
334,23090,sardine,bad,Softness,20230915094934980_sardine_bad.jpeg,2023-09-15
336,23091,sardine,bad,Softness,20230915094945234_sardine_bad.jpeg,2023-09-15
337,23092,sardine,bad,Softness,20230915094957671_sardine_bad.jpeg,2023-09-15
338,23093,sardine,bad,Softness,20230915095009089_sardine_bad.jpeg,2023-09-15
339,23094,sardine,bad,Softness,20230915095019471_sardine_bad.jpeg,2023-09-15
...,...,...,...,...,...,...
22219,70299,sardine,bad,Softness,20240820132110081_sardine_bad.jpeg,2024-08-20
22220,70300,sardine,bad,Softness,20240820132122632_sardine_bad.jpeg,2024-08-20
22221,70301,sardine,bad,Softness,20240820132124485_sardine_bad.jpeg,2024-08-20
22222,70302,sardine,bad,Softness,20240820132135641_sardine_bad.jpeg,2024-08-20


In [ ]:
filtered_df[(filtered_df['Date']>'2024-04-30')&(filtered_df['description']=='Softness')]

,id,type,labels,description,Filename,Date
10749,65837,sardine,bad,Softness,20240525102859293_sardine_bad.jpeg,2024-05-25
10750,65842,sardine,bad,Softness,20240525103024674_sardine_bad.jpeg,2024-05-25
10751,65844,sardine,bad,Softness,20240525103059972_sardine_bad.jpeg,2024-05-25
10752,65846,sardine,bad,Softness,20240525103116442_sardine_bad.jpeg,2024-05-25
10753,65848,sardine,bad,Softness,20240525103156975_sardine_bad.jpeg,2024-05-25
...,...,...,...,...,...,...
22219,70299,sardine,bad,Softness,20240820132110081_sardine_bad.jpeg,2024-08-20
22220,70300,sardine,bad,Softness,20240820132122632_sardine_bad.jpeg,2024-08-20
22221,70301,sardine,bad,Softness,20240820132124485_sardine_bad.jpeg,2024-08-20
22222,70302,sardine,bad,Softness,20240820132135641_sardine_bad.jpeg,2024-08-20


## Creating Reason based dataset from S3 daily data folder

In [ ]:
'''
This code filters a DataFrame for rows where the 'Date' is after '2024-04-30' and the 'description' is 'Softness'. It then defines functions to copy files from a source folder to a destination folder based on the filtered DataFrame. The code iterates through directories in a specified S3 daily data folder, copying relevant files to a designated destination folder while preserving the directory structure.
'''

ndf = filtered_df[(filtered_df['Date']>'2024-04-30')&(filtered_df['description']=='Softness')]

import shutil
import os
from tqdm import tqdm

# def copy_files2(src_folder, des_folder, ndf):
#     for filename in ndf['Filename'].values:
#         if filename in os.listdir(des_folder): continue
#         new_dest = os.path.join(des_folder, ndf[ndf['Filename']==filename]['description'].values[0])
#         if not os.path.exists(new_dest):
#             os.makedirs(new_dest)
#         shutil.copy(os.path.join(src_folder, filename), new_dest)

def copy_files(src_folder, des_folder, ndf):
    for filename in os.listdir(src_folder):
        if filename in ndf['Filename'].values:
            if filename in os.listdir(des_folder): continue
            shutil.copy(os.path.join(src_folder, filename), des_folder)
        
s3_daily_data_folder = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\S3 Daily Data"
des_folder = r"C:\Users\sowmy\Downloads\sardine softness data\sardine\Bad\Softness"

for date in tqdm(os.listdir(s3_daily_data_folder)):
    folder_path = os.path.join(s3_daily_data_folder, date, 'Sardine', 'Bad', 'Single')
    if not os.path.exists(folder_path): continue
    copy_files(folder_path, des_folder, ndf)

100%|██████████| 146/146 [00:32<00:00,  4.46it/s] 


In [ ]:
ndf

,Unnamed: 0,id,type,labels,description,image_url,Filename,Date
10737,52488,65837,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240525102859293_sardine_bad.jpeg,2024-05-25
10738,52489,65842,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240525103024674_sardine_bad.jpeg,2024-05-25
10739,52490,65844,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240525103059972_sardine_bad.jpeg,2024-05-25
10740,52491,65846,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240525103116442_sardine_bad.jpeg,2024-05-25
10741,52492,65848,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240525103156975_sardine_bad.jpeg,2024-05-25
...,...,...,...,...,...,...,...,...
20963,66673,66750,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240604112101180_sardine_bad.jpeg,2024-06-04
20964,66674,66751,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240604112114425_sardine_bad.jpeg,2024-06-04
20965,66675,66752,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240604112123636_sardine_bad.jpeg,2024-06-04
20966,66676,66753,sardine,bad,Softness,https://fish-data-collection.s3.ap-south-1.ama...,20240604112228342_sardine_bad.jpeg,2024-06-04


## Download reason based dataset directly from S3

'fish-data-collection'

In [ ]:
'''
This code downloads images from an S3 bucket based on a filtered DataFrame. It defines a function to download images to a specified destination folder, preserving the directory structure. The code iterates through a list of dates, filters the DataFrame for each date, and downloads the corresponding images to the destination folder.
'''

'\nThis code downloads images from an S3 bucket based on a filtered DataFrame. It defines a function to download images to a specified destination folder, preserving the directory structure. The code iterates through a list of dates, filters the DataFrame for each date, and downloads the corresponding images to the destination folder.\n'

In [ ]:
import boto3
import os

os.environ['AWS_ACCESS_KEY_ID'] = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''

In [ ]:
from urllib.parse import urlparse

def download_object_from_s3(url, destination_folder,
                            bucket_name='fish-data-collection'):
    # Parse the URL to extract bucket name and object key
    parsed_url = urlparse(url)
    object_key = parsed_url.path[1:]  # Remove the leading '/'
    filename = os.path.join(destination_folder, os.path.basename(object_key))
    # print(f'Parsed URL: {parsed_url}\nBucket name: {bucket_name}\nObject key: {object_key}\n')

    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)
    if os.path.exists(filename):
        return

    # Initialize a Boto3 S3 client
    s3 = boto3.client('s3')

    # Download the object
    try:
        response = s3.download_file(Bucket = bucket_name,
                                    Key = object_key,
                                    Filename = filename)
        # print('Object downloaded successfully.')
    except Exception as e:
        print('Error:', str(e))


In [ ]:
def classify_and_move_images2(df, destination_folder):
    '''
    This function classifies and moves images based on their 'type', 'labels', and 'description' columns in the provided DataFrame. It creates a directory structure in the specified destination folder and downloads the images from S3 to the appropriate directories.
    the final structure will be: destination_folder/type/labels/description/image_filename
    '''
    
    for _, row in df.iterrows():
        image_url = row['image_url']
        image_type = row['type']
        image_label = row['labels']
        image_description = row['description']

        # Create destination directories if they don't exist
        type_folder = os.path.join(destination_folder, image_type)
        labels_folder = os.path.join(type_folder, image_label)
        description_folder = os.path.join(labels_folder, image_description)

        os.makedirs(description_folder, exist_ok=True)

        # Download the image from the provided URL
        download_object_from_s3(image_url, description_folder)

In [ ]:
destination_folder = r"sardine softness data"
# classify_and_move_images2(df, destination_folder)
classify_and_move_images2(filtered_df.loc[16629:], destination_folder)

## Data Analysis

In [ ]:
new_df = df[['type', 'labels', 'description']]

In [ ]:
new_df

,type,labels,description
21448,white prawns,bad,Softness
21452,white prawns,good,Softness
21453,white prawns,bad,Softness
21454,white prawns,bad,Softness
21455,white prawns,bad,Softness
...,...,...,...
23808,sardine,bad,Softness
23809,sardine,bad,"Cuts and Damage, Softness"
23810,sardine,bad,Softness
23811,sardine,bad,"Cuts and Damage, Softness"


In [ ]:
# Create a new DataFrame with unique descriptions for each combination of type and labels
description_df = new_df.groupby(['type', 'labels'])['description'].apply(lambda x: list(set(x))).reset_index()
description_df

,type,labels,description
0,mackerel,bad,"[Cuts and Damage, Softness]"
1,mackerel,good,[Softness]
2,pink perch,bad,[Softness]
3,red prawns,bad,"[Cuts and Damage, Softness, Cuts and Damage, S..."
4,sardine,bad,"[Cuts and Damage, Softness, Cuts and Damage, S..."
5,seer,bad,[Softness]
6,seer,good,[Softness]
7,tiger prawns,bad,[Softness]
8,white pomfret,bad,[Softness]
9,white prawns,bad,"[Cuts and Damage, Softness, Red Head in Prawns..."


In [ ]:
description_df

,type,labels,description
0,mackerel,bad,"[Cuts and Damage, Softness]"
1,mackerel,good,[Softness]
2,pink perch,bad,[Softness]
3,red prawns,bad,"[Cuts and Damage, Softness, Cuts and Damage, S..."
4,sardine,bad,"[Cuts and Damage, Softness, Cuts and Damage, S..."
5,seer,bad,[Softness]
6,seer,good,[Softness]
7,tiger prawns,bad,[Softness]
8,white pomfret,bad,[Softness]
9,white prawns,bad,"[Cuts and Damage, Softness, Red Head in Prawns..."


In [ ]:
description_df['description'][6]

['Softness']

In [ ]:
# Define the function
def process_description(description):
    l = []
    for i in description:
        nl = []
        nl.append(i)
        l.append(tuple(nl))
    return l

In [ ]:
process_description(description_df['description'][6])

[('Softness',)]

In [ ]:
# Apply the function to the 'description' column and store the result in a new column 'processed_description'
description_df['processed_description'] = description_df['description'].apply(process_description)
description_df

,type,labels,description,processed_description
0,mackerel,bad,"[Cuts and Damage, Softness]","[(Cuts and Damage,), (Softness,)]"
1,mackerel,good,[Softness],"[(Softness,)]"
2,pink perch,bad,[Softness],"[(Softness,)]"
3,red prawns,bad,"[Cuts and Damage, Softness, Cuts and Damage, S...","[(Cuts and Damage,), (Softness, Cuts and Damag..."
4,sardine,bad,"[Cuts and Damage, Softness, Cuts and Damage, S...","[(Cuts and Damage, Softness,), (Cuts and Damag..."
5,seer,bad,[Softness],"[(Softness,)]"
6,seer,good,[Softness],"[(Softness,)]"
7,tiger prawns,bad,[Softness],"[(Softness,)]"
8,white pomfret,bad,[Softness],"[(Softness,)]"
9,white prawns,bad,"[Cuts and Damage, Softness, Red Head in Prawns...","[(Cuts and Damage, Softness,), (Red Head in Pr..."
